In [1]:
!nvidia-smi

Tue May 19 20:02:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 555.42.02              Driver Version: 555.42.02      CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        Off |   00000000:01:00.0 Off |                  N/A |
|  0%   33C    P8             12W /  350W |    1605MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("HF_ENDPOINT:", os.environ.get("HF_ENDPOINT"))
print("HF_HUB_DISABLE_XET:", os.environ.get("HF_HUB_DISABLE_XET"))

HF_ENDPOINT: https://hf-mirror.com
HF_HUB_DISABLE_XET: 1


In [ ]:
MODEL_NAME   = "bert-base-uncased"
DATA_DIR     = "/home/yangdejin/nlpcc/nlpcc_task2/data"
OUTPUT_DIR   = "/home/yangdejin/nlpcc/nlpcc_task2/outputs/encoder"
LOGGING_DIR = "/home/yangdejin/nlpcc/nlpcc_task2/logs/bert_rdrop"

NUM_EPOCHS = 5
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
WARMUP_STEPS = 500
WEIGHT_DECAY = 0.01
LOGGING_STEPS = 100
SEED = 3407

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOGGING_DIR, exist_ok=True)
print(f"Model : {MODEL_NAME}")
print(f"Data  : {DATA_DIR}")
print(f"Output: {OUTPUT_DIR}")

Model : bert-base-uncased
Data  : /home/yangdejin/nlpcc/nlpcc_task2/data
Output: /home/yangdejin/nlpcc/nlpcc_task2/outputs/encoder


In [7]:
# Cell 3: 标签定义 + 类别分布统计
import json
from collections import Counter

VALUE_LABELS = [
    "Self-direction–thought",
    "Self-direction–action",
    "Stimulation",
    "Hedonism",
    "Achievement",
    "Power–dominance",
    "Power–resources",
    "Face",
    "Security–personal",
    "Security–societal",
    "Tradition",
    "Conformity–rules",
    "Conformity–interpersonal",
    "Humility",
    "Benevolence–dependability",
    "Benevolence–caring",
    "Universalism–concern",
    "Universalism–nature",
    "Universalism–tolerance",
]

NUM_CLASSES = len(VALUE_LABELS)
label2id = {v: i for i, v in enumerate(VALUE_LABELS)}
id2label = {i: v for i, v in enumerate(VALUE_LABELS)}

print(f"Number of classes: {NUM_CLASSES}")

train_vals = []
with open(os.path.join(DATA_DIR, "train.jsonl")) as f:
    for line in f:
        train_vals.append(json.loads(line)["Value"])

counter = Counter(train_vals)
print("\nTrain class distribution:")
for label in VALUE_LABELS:
    print(f"  {label:<40s}: {counter[label]}")

Number of classes: 19

Train class distribution:
  Self-direction–thought                  : 119
  Self-direction–action                   : 124
  Stimulation                             : 400
  Hedonism                                : 164
  Achievement                             : 174
  Power–dominance                         : 156
  Power–resources                         : 237
  Face                                    : 258
  Security–personal                       : 202
  Security–societal                       : 70
  Tradition                               : 90
  Conformity–rules                        : 385
  Conformity–interpersonal                : 236
  Humility                                : 100
  Benevolence–dependability               : 189
  Benevolence–caring                      : 317
  Universalism–concern                    : 160
  Universalism–nature                     : 71
  Universalism–tolerance                  : 68


In [8]:
# Cell 4: 数据加载 + 文本拼接
import json
import torch
import numpy as np
from datasets import Dataset

def build_text(row):
    parts = []
    if row.get("Scenario"):
        parts.append("Scenario: " + row["Scenario"].strip())
    if row.get("Question"):
        parts.append("Question: " + row["Question"].strip())
    if row.get("Consistent Value Response"):
        parts.append("Response: " + row["Consistent Value Response"].strip())
    return " [SEP] ".join(parts)

def load_jsonl(path, has_label=True):
    texts, labels = [], []
    with open(path) as f:
        for line in f:
            row = json.loads(line)
            texts.append(build_text(row))
            if has_label:
                labels.append(label2id[row["Value"]])
    if has_label:
        return Dataset.from_dict({"text": texts, "label": labels})
    return Dataset.from_dict({"text": texts})

train_dataset = load_jsonl(os.path.join(DATA_DIR, "train.jsonl"))
dev_dataset   = load_jsonl(os.path.join(DATA_DIR, "dev.jsonl"))

print(f"Train size : {len(train_dataset)}")
print(f"Dev size   : {len(dev_dataset)}")
print(f"\nSample text:\n{train_dataset[0]['text'][:300]}")

Train size : 3520
Dev size   : 514

Sample text:
Scenario: You work in a corporate setting where your manager frequently imposes strict guidelines. [SEP] Question: How would you handle a disagreement with a superior during a team meeting? [SEP] Response: I would suggest alternative solutions diplomatically, ensuring my input is heard without chall


In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertTokenizerFast, DataCollatorWithPadding
from transformers import BertPreTrainedModel, BertModel
from transformers.modeling_outputs import SequenceClassifierOutput

class BertScratch(BertPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config
        self.bert = BertModel(config)
        classifier_dropout = (
            config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)
        self.post_init()
    def forward(self, input_ids=None, attention_mask=None, token_type_ids = None, labels=None):
        outputs = self.bert(input_ids, attention_mask, token_type_ids)
        pooled_output = outputs[1]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        kl_outputs = self.bert(input_ids, attention_mask, token_type_ids)
        kl_pooled_output = kl_outputs[1]
        kl_pooled_output = self.dropout(kl_pooled_output)
        kl_logits = self.classifier(kl_pooled_output)

        total_loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            ce_loss = loss_fct(kl_logits.view(-1, self.num_labels), labels.view(-1))
            kl_loss = (KL(logits, kl_logits, "sum") + KL(kl_logits, logits, "sum")) / 2.0
            total_loss = loss + ce_loss + kl_loss
        
        return SequenceClassifierOutput(
            loss = total_loss,
            logits = logits,
            hidden_states = outputs.hidden_states,
            attentions = outputs.attentions,
        )
def KL(input, target, reduction="sum"):
    input = input.float()
    target = target.float()
    loss = F.kl_div(
        F.log_softmax(input, dim = -1, dtype=torch.float32),
        F.softmax(target, dim = -1, dtype=torch.float32),
        reduction = reduction,
    )
    return loss
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

def tokenizer_fn(examples):
    return tokenizer(examples["text"], truncation = True)

train_dataset = train_dataset.map(tokenizer_fn, batched=True)
dev_dataset = dev_dataset.map(tokenizer_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer = tokenizer)
model = BertScratch.from_pretrained(MODEL_NAME, num_labels = NUM_CLASSES)
model.config.id2label = id2label
model.config.label2id = label2id

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")

loading file vocab.txt from cache at /home/yangdejin/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/vocab.txt
loading file tokenizer.json from cache at /home/yangdejin/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/tokenizer.json
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at /home/yangdejin/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/tokenizer_config.json
loading file chat_template.jinja from cache at None
loading configuration file config.json from cache at /home/yangdejin/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/config.json
Model config BertConfig {
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": n

Map:   0%|          | 0/3520 [00:00<?, ? examples/s]

Map:   0%|          | 0/514 [00:00<?, ? examples/s]

loading configuration file config.json from cache at /home/yangdejin/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/config.json
Model config BertConfig {
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2",
    "3": "LABEL_3",
    "4": "LABEL_4",
    "5": "LABEL_5",
    "6": "LABEL_6",
    "7": "LABEL_7",
    "8": "LABEL_8",
    "9": "LABEL_9",
    "10": "LABEL_10",
    "11": "LABEL_11",
    "12": "LABEL_12",
    "13": "LABEL_13",
    "14": "LABEL_14",
    "15": "LABEL_15",
    "16": "LABEL_16",
    "17": "LABEL_17",
    "18": "LABEL_18"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_10": 10,
    "LABEL_11": 1

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

loading weights file model.safetensors from cache at /home/yangdejin/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/model.safetensors
A pretrained model of type `BertScratch` contains parameters that have been renamed internally (a few are listed below but more are present in the model):
* `cls.predictions.transform.LayerNorm.beta` -> `cls.predictions.transform.LayerNorm.bias`
* `cls.predictions.transform.LayerNorm.gamma` -> `cls.predictions.transform.LayerNorm.weight`
If you are using a model from the Hub, consider submitting a PR to adjust these weights and help future users.
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertScratch: ['cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- 

Total parameters: 109496851


In [12]:
# Cell 6: 模型参数统计
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

Trainable parameters: 109,496,851 / 109,496,851 (100.00%)


In [15]:
# Cell 7: 训练
import inspect
import evaluate
import numpy as np
from transformers import TrainingArguments, Trainer

accuracy_metric = evaluate.load("accuracy")
f1_metric       = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1  = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "macro_f1": f1}

training_kwargs = dict(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    logging_dir=LOGGING_DIR,
    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    seed=SEED,
    report_to="none",
)
if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters:
    training_kwargs["eval_strategy"] = "epoch"
else:
    training_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer_stats = trainer.train()
print(trainer_stats)

PyTorch: setting up devices
/tmp/ipykernel_517877/1169971219.py:37: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The following columns in the Training set don't have a corresponding argument in `BertScratch.forward` and have been ignored: text. If text are not expected by `BertScratch.forward`,  you can safely ignore this message.
***** Running training *****
  Num examples = 3,520
  Num Epochs = 5
  Instantaneous batch size per device = 4
  Total train batch size (w. parallel, distributed & accumulation) = 4
  Gradient Accumulation steps = 1
  Total optimization steps = 4,400
  Number of trainable parameters = 109,496,851


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.710200,2.116683,0.844358,0.828761
2,0.558100,1.496859,0.902724,0.887296
3,0.178100,1.275543,0.924125,0.913456
4,0.246200,1.257249,0.922179,0.914620
5,0.082900,1.037258,0.933852,0.922526


The following columns in the Evaluation set don't have a corresponding argument in `BertScratch.forward` and have been ignored: text. If text are not expected by `BertScratch.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 514
  Batch size = 8
The following columns in the Evaluation set don't have a corresponding argument in `BertScratch.forward` and have been ignored: text. If text are not expected by `BertScratch.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 514
  Batch size = 8
The following columns in the Evaluation set don't have a corresponding argument in `BertScratch.forward` and have been ignored: text. If text are not expected by `BertScratch.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 514
  Batch size = 8
The following columns in the Evaluation set don't have a corresponding argument in `BertScratch.forward` and have been ignor

TrainOutput(global_step=4400, training_loss=0.29616317118560387, metrics={'train_runtime': 351.5163, 'train_samples_per_second': 50.069, 'train_steps_per_second': 12.517, 'total_flos': 985402620776064.0, 'train_loss': 0.29616317118560387, 'epoch': 5.0})


In [ ]:
# Cell 8: Dev 集综合评估 + 每类 F1 + 混淆矩阵热图
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

pred_output = trainer.predict(dev_dataset)
preds  = np.argmax(pred_output.predictions, axis=-1)
labels = np.array(dev_dataset["label"])

print("=" * 60)
print("Dev Set Evaluation Results — R-Drop")
print("=" * 60)
print(classification_report(
    labels, preds,
    target_names=VALUE_LABELS,
    digits=4
))

cm = confusion_matrix(labels, preds)
short_labels = [
    "SD-thought", "SD-action", "Stim", "Hedon", "Achiev",
    "Pow-dom", "Pow-res", "Face", "Sec-per", "Sec-soc",
    "Trad", "Conf-rul", "Conf-int", "Hum", "Ben-dep",
    "Ben-car", "Uni-con", "Uni-nat", "Uni-tol"
]

plt.figure(figsize=(14, 12))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=short_labels, yticklabels=short_labels
)
plt.title(f"Confusion Matrix — R-Drop — {MODEL_NAME}")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"), dpi=150)
plt.show()
print(f"Confusion matrix saved to {OUTPUT_DIR}/confusion_matrix.png")